# Sección 5 – Modelos Baseline con StatsForecast (Nixtla)

Este notebook implementa y evalúa la **familia de modelos baseline** usando la librería [StatsForecast](https://nixtla.github.io/statsforecast/) de Nixtla.

**Modelos evaluados:**
- `Naive` – repite el último valor observado
- `SeasonalNaive(season_length=24)` – repite el mismo período del día anterior
- `SeasonalNaive(season_length=168)` – repite la misma hora de la semana anterior
- `MSTL` – descomposición de temporadas múltiples (24 h y 168 h)
- `AutoETS` – suavizamiento exponencial seleccionado automáticamente

**Protocolo:** backtesting con origen móvil diario usando `backtest()` de `src/backtest.py`.

## 0. Configuración e importaciones

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Agregar la raíz del proyecto al path para importar src/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, MSTL, AutoETS

from src.backtest import backtest, ForecastModel
from src.metrics import evaluate_all, mae_per_horizon

# ── Rutas ──────────────────────────────────────────────────────────────────
DATA_PATH    = os.path.join(PROJECT_ROOT, 'data', 'processed', 'panel_hourly.parquet')
OUTPUT_DIR   = os.path.join(PROJECT_ROOT, 'outputs')
FIGURES_DIR  = os.path.join(OUTPUT_DIR, 'figures')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Parámetros globales ────────────────────────────────────────────────────
TRAIN_END  = '2014-09-30'
TEST_START = '2014-10-01'
TEST_END   = '2014-12-31'
H          = 24   # horizonte de pronóstico (horas)

print('Project root:', PROJECT_ROOT)
print('Data path   :', DATA_PATH)

## 1. Carga y exploración del panel

In [ ]:
panel = pd.read_parquet(DATA_PATH)
panel.index = pd.to_datetime(panel.index)
panel = panel.asfreq('h')   # asegurar frecuencia horaria

print(f'Panel shape  : {panel.shape}  (T={panel.shape[0]} horas, N={panel.shape[1]} clientes)')
print(f'Rango temporal: {panel.index[0]}  →  {panel.index[-1]}')
print(f'NaN totales  : {panel.isna().sum().sum()}')
panel.head(3)

In [ ]:
# ── Separar entrenamiento y prueba ─────────────────────────────────────────
panel_train = panel.loc[:TRAIN_END]
panel_test  = panel.loc[TEST_START:TEST_END]

# Array y_train para MASE (N, T_train)
y_train = panel_train.values.T

print(f'Entrenamiento : {panel_train.shape}  ({panel_train.index[0].date()} – {panel_train.index[-1].date()})')
print(f'Prueba        : {panel_test.shape}   ({panel_test.index[0].date()} – {panel_test.index[-1].date()})')

## 2. Transformación al formato Nixtla

StatsForecast requiere un DataFrame largo con columnas `(unique_id, ds, y)`.

In [ ]:
def panel_to_nixtla(df: pd.DataFrame, value_name: str = 'y') -> pd.DataFrame:
    """
    Convierte panel (T x N) → formato Nixtla (unique_id, ds, y).

    Args:
        df         : DataFrame con índice datetime y columnas = clientes.
        value_name : nombre de la columna de valores.

    Returns:
        DataFrame largo con columnas [unique_id, ds, y].
    """
    nixtla_df = (
        df
        .reset_index()                          # índice → columna 'index'
        .rename(columns={df.index.name or 'index': 'ds'})
        .melt(id_vars='ds', var_name='unique_id', value_name=value_name)
        [['unique_id', 'ds', value_name]]       # reordenar
        .sort_values(['unique_id', 'ds'])
        .reset_index(drop=True)
    )
    return nixtla_df


# Transformar el panel completo una vez
nixtla_full = panel_to_nixtla(panel)
nixtla_train = panel_to_nixtla(panel_train)

print(f'Filas en formato Nixtla (entrenamiento): {len(nixtla_train):,}')
print(f'Clientes únicos: {nixtla_train["unique_id"].nunique()}')
nixtla_train.head()

## 3. Wrapper `StatsForecastModel` (protocolo `ForecastModel`)

El wrapper adapta la API de StatsForecast al protocolo `predict(history, horizon, origin_time)` requerido por `backtest()`.

In [ ]:
class StatsForecastModel:
    """
    Wrapper que adapta StatsForecast al protocolo ForecastModel de src/backtest.py.

    Crea internamente un objeto StatsForecast con el modelo dado,
    lo ajusta sobre el history disponible en cada origen y retorna
    un array (N_clientes, H) con las predicciones.
    """

    def __init__(self, sf_model, model_name: str, n_jobs: int = -1):
        """
        Args:
            sf_model   : instancia de un modelo StatsForecast (Naive, SeasonalNaive, etc.).
            model_name : nombre del modelo para reportes.
            n_jobs     : número de workers paralelos (-1 = todos los núcleos).
        """
        self._sf_model  = sf_model
        self._name      = model_name
        self._n_jobs    = n_jobs

    @property
    def name(self) -> str:
        return self._name

    def predict(
        self,
        history: pd.DataFrame,
        horizon: int,
        origin_time: pd.Timestamp,
    ) -> np.ndarray:
        """
        Ajusta el modelo sobre `history` y produce pronósticos.

        Args:
            history     : DataFrame (T x N), datos disponibles hasta cutoff.
            horizon     : número de pasos (H = 24).
            origin_time : instante del pronóstico (sin uso directo aquí).

        Returns:
            array (N, H) con predicciones en kWh.
        """
        clients = list(history.columns)

        # ── Convertir al formato Nixtla ─────────────────────────────────────
        nixtla_df = panel_to_nixtla(history)

        # Eliminar NaN (StatsForecast no los acepta)
        nixtla_df = nixtla_df.dropna(subset=['y'])

        # ── Crear y ajustar StatsForecast ───────────────────────────────────
        sf = StatsForecast(
            models=[self._sf_model],
            freq='h',
            n_jobs=self._n_jobs,
            verbose=False,
        )
        forecast_df = sf.forecast(df=nixtla_df, h=horizon)

        # ── Pivotar a (N, H) ────────────────────────────────────────────────
        # forecast_df tiene columnas: [unique_id, ds, <model_alias>]
        model_col = [c for c in forecast_df.columns if c not in ('unique_id', 'ds')][0]
        forecast_df = forecast_df.reset_index() if 'unique_id' in forecast_df.index.names else forecast_df

        pivot = (
            forecast_df
            .sort_values(['unique_id', 'ds'])
            .pivot(index='unique_id', columns='ds', values=model_col)
            .reindex(clients)   # mantener el orden original de clientes
        )

        preds = pivot.values  # (N, H)

        # Clip a 0 (consumo no puede ser negativo)
        preds = np.clip(preds, 0, None)

        # Asegurar shape (N, H) aunque haya clientes sin datos
        if preds.shape[1] < horizon:
            pad = np.zeros((preds.shape[0], horizon - preds.shape[1]))
            preds = np.hstack([preds, pad])

        return preds

## 4. Backtesting de los 5 modelos baseline

> **Nota:** Cada origen re-ajusta el modelo con datos acumulados hasta el cutoff.  
> Para MSTL y AutoETS esto puede tardar varios minutos dependiendo del hardware.

In [ ]:
# ── Definición de modelos ───────────────────────────────────────────────────
# Ajustar hiperparámetros aquí si es necesario
MODELS = [
    StatsForecastModel(
        sf_model=Naive(),
        model_name='Naive',
    ),
    StatsForecastModel(
        sf_model=SeasonalNaive(season_length=24),   # periodo diario
        model_name='SeasonalNaive_24',
    ),
    StatsForecastModel(
        sf_model=SeasonalNaive(season_length=168),  # periodo semanal
        model_name='SeasonalNaive_168',
    ),
    StatsForecastModel(
        sf_model=MSTL(
            season_length=[24, 168],  # temporadas diaria y semanal
            # trend_forecaster='auto_arima',  # descomenta para usar ARIMA en tendencia
        ),
        model_name='MSTL',
    ),
    StatsForecastModel(
        sf_model=AutoETS(
            season_length=24,   # temporada principal
            # model='ZZZ',      # selección automática de error/tendencia/estacionalidad
        ),
        model_name='AutoETS',
    ),
]

print('Modelos a evaluar:', [m.name for m in MODELS])

In [ ]:
# ── Ejecutar backtest para cada modelo ────────────────────────────────────
results = {}

for model in MODELS:
    print(f'\n{'='*60}')
    print(f'  Evaluando: {model.name}')
    print(f'{'='*60}')

    res = backtest(
        model=model,
        panel=panel,
        y_train=y_train,
        H=H,
        test_start=TEST_START,
        test_end=TEST_END,
        seasonality=24,
        return_predictions=True,
        verbose=True,
    )
    results[model.name] = res
    print(f'  → MAE: {res["metrics"]["MAE"]:.4f} kWh  |  Tiempo: {res["inference_time"]:.1f}s')

print('\n✓ Backtest completo.')

## 5. Guardar métricas y predicciones

In [ ]:
# ── 5.1 Métricas globales ──────────────────────────────────────────────────
metrics_rows = [res['metrics'] for res in results.values()]
metrics_df   = pd.DataFrame(metrics_rows)

# Agregar columna de modelo si no está
if 'model' not in metrics_df.columns:
    metrics_df.insert(0, 'model', list(results.keys()))

metrics_path = os.path.join(OUTPUT_DIR, 'metrics_baselines.csv')
metrics_df.to_csv(metrics_path, index=False)
print(f'Métricas guardadas en: {metrics_path}')
metrics_df

In [ ]:
# ── 5.2 Predicciones ──────────────────────────────────────────────────────
all_predictions = pd.concat(
    [res['predictions'] for res in results.values() if res['predictions'] is not None],
    ignore_index=True,
)

# Asegurar columnas requeridas: (unique_id, ds, modelo, y_hat)
all_predictions = all_predictions[['unique_id', 'ds', 'modelo', 'y_hat']]

preds_path = os.path.join(OUTPUT_DIR, 'predictions_baselines.csv')
all_predictions.to_csv(preds_path, index=False)
print(f'Predicciones guardadas en: {preds_path}')
print(f'Filas totales: {len(all_predictions):,}')
all_predictions.head()

## 6. Visualización – MAE por paso del horizonte (h=1…24)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

COLORS = {
    'Naive'              : '#e74c3c',
    'SeasonalNaive_24'   : '#3498db',
    'SeasonalNaive_168'  : '#2ecc71',
    'MSTL'               : '#9b59b6',
    'AutoETS'            : '#f39c12',
}

MARKERS = ['o', 's', '^', 'D', 'v']

for (model_name, res), marker in zip(results.items(), MARKERS):
    mae_h = res['mae_horizon']   # array (H,)
    ax.plot(
        range(1, H + 1),
        mae_h,
        label=model_name,
        color=COLORS.get(model_name),
        marker=marker,
        markersize=5,
        linewidth=1.8,
    )

ax.set_xlabel('Paso del horizonte (h)', fontsize=12)
ax.set_ylabel('MAE (kWh)', fontsize=12)
ax.set_title('MAE por paso del horizonte — Modelos Baseline', fontsize=14, fontweight='bold')
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(loc='upper left', framealpha=0.9)

fig.tight_layout()
fig_path = os.path.join(FIGURES_DIR, 'mae_horizon_baselines.png')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Figura guardada en: {fig_path}')
plt.show()

## 7. Análisis del error en la semana 22-28 de diciembre

El período 22-28 de diciembre corresponde a la semana de Navidad — típicamente atípica por feriados, cambios de consumo y reducción de actividad industrial.

In [ ]:
# ── Filtrar predicciones en la semana 22-28 de diciembre ──────────────────
all_predictions['ds'] = pd.to_datetime(all_predictions['ds'])
mask_xmas = (
    (all_predictions['ds'].dt.month == 12) &
    (all_predictions['ds'].dt.day >= 22) &
    (all_predictions['ds'].dt.day <= 28)
)
preds_xmas = all_predictions[mask_xmas].copy()

# ── Añadir valores reales ──────────────────────────────────────────────────
panel_long = panel_to_nixtla(panel, value_name='y_real')
panel_long['ds'] = pd.to_datetime(panel_long['ds'])

preds_xmas = preds_xmas.merge(panel_long, on=['unique_id', 'ds'], how='left')
preds_xmas['abs_error'] = (preds_xmas['y_hat'] - preds_xmas['y_real']).abs()

# ── MAE por modelo en la semana navideña ──────────────────────────────────
xmas_metrics = (
    preds_xmas
    .groupby('modelo')['abs_error']
    .agg(MAE='mean', RMSE=lambda x: np.sqrt((x**2).mean()), Count='count')
    .reset_index()
    .sort_values('MAE')
)
print('\n📅 Error en semana navideña (22-28 dic):')
xmas_metrics

In [ ]:
# ── Comparar MAE global vs. semana navideña ────────────────────────────────
global_mae = metrics_df.set_index('model')['MAE'].rename('MAE_global')
xmas_mae   = xmas_metrics.set_index('modelo')['MAE'].rename('MAE_xmas')

comparison = pd.concat([global_mae, xmas_mae], axis=1)
comparison['delta_%'] = ((comparison['MAE_xmas'] - comparison['MAE_global']) / comparison['MAE_global'] * 100).round(2)

print('\n📊 Comparativa MAE global vs. semana navideña:')
print(comparison.to_string())

In [ ]:
# ── Gráfico: error horario promedio durante la semana navideña ─────────────
preds_xmas['hour'] = preds_xmas['ds'].dt.hour
hourly_error_xmas = preds_xmas.groupby(['modelo', 'hour'])['abs_error'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))

for model_name, grp in hourly_error_xmas.groupby('modelo'):
    ax.plot(
        grp['hour'], grp['abs_error'],
        label=model_name,
        color=COLORS.get(model_name),
        linewidth=1.8,
    )

ax.set_xlabel('Hora del día', fontsize=12)
ax.set_ylabel('MAE promedio (kWh)', fontsize=12)
ax.set_title('Error horario — Semana Navideña (22-28 Dic)', fontsize=14, fontweight='bold')
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(loc='upper left')
fig.tight_layout()

fig_path_xmas = os.path.join(FIGURES_DIR, 'mae_xmas_week_baselines.png')
fig.savefig(fig_path_xmas, dpi=150, bbox_inches='tight')
print(f'Figura guardada en: {fig_path_xmas}')
plt.show()

## 8. Resumen final

In [ ]:
print('='*65)
print('RESUMEN – Modelos Baseline')
print('='*65)
print(metrics_df.to_string(index=False))
print()
print(f'Mejor modelo (MAE global): {metrics_df.loc[metrics_df["MAE"].idxmin(), "model"]}')
print()
print('Archivos generados:')
print(f'  → {preds_path}')
print(f'  → {metrics_path}')
print(f'  → {fig_path}')
print(f'  → {fig_path_xmas}')